In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd 


C:\Users\wasee\anaconda3\envs\nlp_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
processed_df = pd.read_csv("../../datasets/raw/semienz_jobs_description_classified.csv")

# Handling duplicates 

In [3]:
len(processed_df[processed_df.duplicated()])

5

In [4]:
processed_df = processed_df.drop_duplicates()

In [15]:
# remove irelvant cols
processed_df.drop(columns=["description","clean_description","description_en"],inplace=True)

In [24]:
processed_df.head()

,job_id,job_title,posted_since,organization,field_of_work,company,experience_level,job_type,work_mode,employment_type,location,new_description,clean_description,final_job_description
0,503010,Service Architect for network services,2026-05-05,Information Technology,Information Technology,Siemens Technology and Services Private Limited,Experienced Professional,Full-time,Office/Site only,Permanent,Bangalore - Karnataka - India,"Hello Visionary,\nWe empower our people to sta...","Hello Visionary,\nWe empower our people to sta...",hello visionary we empower our people to stay ...
1,503299,"Werkstudent (w/m/d) Marketing, Communication &...",2026-05-05,Global Business Services,Internal Services,Siemens AG,Student (Not Yet Graduated),Part-time,Office/Site only,Fixed Term,Mannheim - Baden-Wuerttemberg - Germany,Become part of our team as a working student (...,Become part of our team as a working student (...,become part of our team as a working student f...
2,503374,Site Reliability Engineer,2026-05-05,Foundational Technologies,Research & Development,Siemens Technology and Services Private Limited,Experienced Professional,Full-time,Office/Site only,Permanent,Bangalore - Karnataka - India,We empower our people to stay resilient and re...,We empower our people to stay resilient and re...,we empower our people to stay resilient and re...
8,503133,Data Scientist & Data Product Manager in MR (f...,2026-05-06,Siemens Healthineers,Research & Development,Siemens Healthineers AG,Mid-level Professional,Full-time/Part-time,Hybrid (Remote/Office),Permanent,Erlangen - - Germany,"Our name, Siemens Healthineers, was selected t...","Our name, Siemens Healthineers, was selected t...",our name siemens healthineers was selected to ...
9,503134,Data Strategist & Data Architect in MR (f/m/d),2026-05-06,Siemens Healthineers,Research & Development,Siemens Healthineers AG,Experienced Professional,Full-time/Part-time,Hybrid (Remote/Office),Permanent,Erlangen - - Germany,"Our name, Siemens Healthineers, was selected t...","Our name, Siemens Healthineers, was selected t...",our name siemens healthineers was selected to ...


# Missing value Handling 

In [6]:
# job_id 
processed_df.job_id.isna().sum() 
processed_df.job_id.dtype

dtype('int64')

In [7]:
# job title 
processed_df.job_title.isna().sum()
processed_df.job_title.apply(lambda x: len(x.split()) < 2)
processed_df.job_title.value_counts() 
processed_df.job_title.nunique()

1742

In [8]:
processed_df.organization.isna().sum()
processed_df.organization.apply(lambda x: len(x.split()) < 2)
processed_df.organization.value_counts() 


organization
Smart Infrastructure                                 754
Digital Industries                                   284
Mobility                                             250
Siemens Healthineers                                 220
Siemens Energy                                       101
Foundational Technologies                            100
Global Business Services                              94
Information Technology                                21
Controlling and Finance                               18
People & Organization                                 14
Siemens Financial Services                            12
Siemens Real Estate                                   12
Country Functions & Departments                       11
Advanta                                               11
Cybersecurity                                          8
Legal, Compliance and Intellectual Property            7
Supply Chain Management                                6
Data & Artificial 

In [9]:
# we have missing values in new_description col
len(processed_df[processed_df.new_description.isna()])

6

In [10]:
processed_df = processed_df.dropna(subset=["new_description"])

# Handle size of description and Language issues 

In [11]:
filtered_df = processed_df[ processed_df["new_description"].apply(
        lambda x: len(str(x).split()) < 20 
    ) ]


In [12]:
filtered_df

,job_id,job_title,posted_since,organization,field_of_work,company,experience_level,job_type,work_mode,employment_type,location,description,clean_description,description_en,new_description
735,503251,智慧配电销售 （厦门）,2026-04-17,Smart Infrastructure,Sales,"Siemens Ltd., China",Mid-level Professional,Full-time,Office/Site only,Fixed Term,Xiamen - Fujian Sheng - China,智慧配电销售\n零碳先锋，美丽中国！\n西门子智能基础设施\n(Smart Infrastr...,smart infrastructure si si,智慧配电销售\n零碳先锋，美丽中国！\n西门子智能基础设施\n(Smart Infrastr...,智慧配电销售\n零碳先锋，美丽中国！\n西门子智能基础设施\n(Smart Infrastr...
1177,504148,电子半导体行业大客户经理,2026-04-24,Digital Industries,Sales,"Siemens Ltd., China",Mid-level Professional,Full-time,Office/Site only,Fixed Term,NaN,加入西门子数字化工业集团，共创明日世界！\n西门子数字化工业集团\n(Digital Ind...,digital industries di 5 15 2019 46 2030 2019 9...,加入西门子数字化工业集团，共创明日世界！\n西门子数字化工业集团\n(Digital Ind...,2030\n年的二氧化碳减排目标为相比\n2019\n年减少\n90%\n。在中国，西门子入...


In [13]:
processed_df= processed_df[~processed_df.index.isin(filtered_df.index)]

In [14]:
processed_df.shape

(1925, 15)

## Translate to Engish if any other languae is there

In [16]:
from langdetect import detect
from deep_translator import GoogleTranslator

MAX_CHARS = 4500  # stay below the limit

def chunk_text(text, max_chars=MAX_CHARS):
    return [
        text[i:i + max_chars]
        for i in range(0, len(text), max_chars)
    ]

def translate_to_english(text):
    try:
        lang = detect(text)

        if lang == "en":
            return text

        chunks = chunk_text(text)

        translated_chunks = [
            GoogleTranslator(source=lang, target="en").translate(chunk)
            for chunk in chunks
        ]

        return " ".join(translated_chunks)

    except Exception as e:
        print(f"Translation error: {e}")
        return text

In [ ]:
processed_df["clean_description"] = processed_df["new_description"].apply(translate_to_english)

## Remove the Chinies language

In [19]:
chinies_jobs = processed_df[processed_df.clean_description.apply(lambda x:detect(x)=="zh-cn")]

In [20]:
processed_df = processed_df[
    ~processed_df.index.isin(chinies_jobs.index)
]

# Cleaning Text data

In [33]:
import re
import contractions

def clean_text(text):
    
    text = contractions.fix(text)
    # remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # remove urls
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    text = str(text).lower()  # lowercase
    
    text = re.sub(r'\n', ' ', text)          # remove new lines
    text = re.sub(r'\r', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)         # remove extra spaces
    return text.strip()


In [34]:
processed_df["final_job_description"]=processed_df.clean_description.apply(clean_text)

In [ ]:
processed_df.final_job_description.to_list()[120:130]

## Remove company boilerplate

In [46]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

def remove_company_boilerplate(text: str) -> str:
    """
    Remove generic company branding, DEI statements, talent community,
    and marketing sentences from job descriptions.
    """

    patterns = [
        # diversity / equal opportunity
        r"\bequal opportunities\b",
        r"\bwe welcome applications from\b",
        r"\bpeople with disabilities\b",
        r"\bdifferent backgrounds\b",

        # career information
        r"\bjobs and careers at\b", 
        r"\bjobs careers at siemens\b"
        r"\bmore information about jobs\b",

        # talent community / alerts
        r"\btalent community\b",
        r"\bpersonalized job alert",
        r"\bnew opportunities\b",
        r"\bnot need superheroes\b", 
        r"\bemployment decisions at siemens\b",
        
        # CV / resume by email
        r"\bdo not send us your cv\b",
        r"\bdo not send us your resume\b",
        r"\bresume by email\b",

        # unsolicited applications
        r"\bunsolicited information\b",
        r"\bdelete and destroy unsolicited\b",
        r"\brefrain from any such practices\b",

        # marketing / branding
        r"\btransform the everyday with us\b",
        r"\bbring your curiosity\b",
        r"\bhelp us shape tomorrow\b",
        r"\bshape tomorrow\b",
        r"\bhelp us develop sustainable future products\b",
        r"\bshaping the transition to a sustainable energy system\b",
        r"\breliable and affordable energy supply\b",

        # company introduction
        r"\babout us\b",
        r"\bready to actively shape the future\b",
        r"\bsmart infrastructure buildings division\b",
        r"\bmaking buildings smarter\b",
        r"\bsafer and more sustainable\b",

        # policy
        r"\byour adherence to our policies\b",
        r"\bis appreciated\b",
    ]

    docs = nlp(text)

    cleaned_sentences = []

    for sent in docs.sents:
        s = sent.text.strip()

        remove = any(
            re.search(pattern, s, flags=re.I)
            for pattern in patterns
        )

        if not remove:
            cleaned_sentences.append(s)

    return " ".join(cleaned_sentences)

In [48]:
processed_df["final_job_description"]=processed_df.final_job_description.apply(remove_company_boilerplate)

In [ ]:
for des in processed_df.final_job_description.to_list()[120:130]: 
    print("---------------Orignal------------------\n")
    print(des)
    
    print("-----------------Updated-----------------\n") 
    print(remove_company_boilerplate(des)

In [50]:
processed_df.to_csv("../../datasets/processed/semienz_cleaned.v2.csv")  # load the semiez prepareed data 